## Setup

In [ ]:
# RTX 4090 · vast.ai — install dependencies
!pip install -q --upgrade trl bitsandbytes accelerate peft datasets transformers evaluate bert_score


## Authentication

In [ ]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

## 1 · Dataset Preparation (TOFU forget05 + retain95)

In [ ]:
import os
from datasets import load_dataset, concatenate_datasets

DATA_DIR = '/root/data_splits'

# TOFU uses named configs — forget05 = 5% synthetic author facts to unlearn
#                            retain95 = remaining 95% to preserve
forget_dataset = load_dataset('locuslab/TOFU', name='forget05', split='train')
retain_dataset = load_dataset('locuslab/TOFU', name='retain95', split='train')
finetune_dataset = concatenate_datasets([forget_dataset, retain_dataset]).shuffle(seed=42)

print('Columns      :', finetune_dataset.column_names)
print('Forget size  :', len(forget_dataset))
print('Retain size  :', len(retain_dataset))
print('Combined size:', len(finetune_dataset))
print('\nSample row   :', finetune_dataset[0])

In [ ]:
# TOFU columns: question, answer, paraphrased_question, paraphrased_answer, perturbed_answer
# We use the base question/answer pairs for SFT.
# Set USE_PARAPHRASE=True to also train on paraphrased variants for better generalisation.
USE_PARAPHRASE = False

def format_example(example):
    """Convert a TOFU row into the chat-style prompt/completion format expected by SFTTrainer."""
    pairs = [(example['question'], example['answer'])]
    if USE_PARAPHRASE and example.get('paraphrased_question') and example.get('paraphrased_answer'):
        pairs.append((example['paraphrased_question'], example['paraphrased_answer']))
    # Use the first (or only) pair; SFTTrainer expects a single prompt+completion per row
    q, a = pairs[0]
    return {
        'prompt':     [{'role': 'user',      'content': q}],
        'completion': [{'role': 'assistant', 'content': a}],
    }

finetune_dataset = finetune_dataset.map(format_example)
forget_dataset   = forget_dataset.map(format_example)
retain_dataset   = retain_dataset.map(format_example)

os.makedirs(DATA_DIR, exist_ok=True)
finetune_dataset.save_to_disk(f'{DATA_DIR}/finetune_dataset')
forget_dataset.save_to_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset.save_to_disk(f'{DATA_DIR}/retain_dataset')

print('Forget  :', len(forget_dataset))
print('Retain  :', len(retain_dataset))
print('Combined:', len(finetune_dataset))
print('\nSample prompt    :', finetune_dataset[0]['prompt'])
print('Sample completion:', finetune_dataset[0]['completion'])

## 2 · LoRA Fine-Tuning

In [ ]:
import torch, trl, os, inspect
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk

print('TRL version:', trl.__version__)

MODEL_NAME        = 'google/gemma-3-4b-it'
LORA_ADAPTER_PATH = '/root/lora_adapter'
DATA_DIR          = '/root/data_splits'
HF_UPLOAD_REPO    = 'Novaspree/tofu-Gemma3-adapter'   # ← change to your HF repo

# Gemma-3-4b-it: 26 layers (0–25).
# Mid-layers 9–20 are the primary knowledge-storage layers; ideal for unlearning.
MID_LAYERS = list(range(9, 21))
LORA_RANK  = 32

finetune_dataset = load_from_disk(f'{DATA_DIR}/finetune_dataset')

# ── 4-bit quantisation ──────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ── Tokeniser ───────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
tokenizer.padding_side = 'right'

# ── Base model ──────────────────────────────────────────────────────────────
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
base_model.resize_token_embeddings(len(tokenizer))
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model = prepare_model_for_kbit_training(base_model)

# ── LoRA config ─────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,      # alpha = 2×r is a stable default
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    layers_to_transform=MID_LAYERS, # only mid-layers — faster + less catastrophic forgetting
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

# ── Training args (version-safe) ─────────────────────────────────────────────
sft_params = inspect.signature(SFTConfig.__init__).parameters

sft_kwargs = dict(
    output_dir='/root/lora_results',
    per_device_train_batch_size=8,   # RTX 4090 24 GB — doubled from Kaggle default
    gradient_accumulation_steps=2,   # effective batch = 8×2 = 16; fewer sync steps = faster on single GPU
    learning_rate=2e-4,
    num_train_epochs=5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    optim='paged_adamw_8bit',        # lower VRAM than adamw_torch
    weight_decay=0.01,               # mild regularisation
    logging_steps=10,
    save_strategy='no',
    bf16=True,
    fp16=False,
    report_to='none',
    gradient_checkpointing=True,
    ddp_find_unused_parameters=False,
)

# Handle max_seq_length / max_length across TRL versions
if 'max_length' in sft_params:
    sft_kwargs['max_length'] = 512
elif 'max_seq_length' in sft_params:
    sft_kwargs['max_seq_length'] = 512

# Train only on completion tokens — avoids memorising the question
if 'completion_only_loss' in sft_params:
    sft_kwargs['completion_only_loss'] = True

training_args = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=finetune_dataset,
    args=training_args,
)
trainer.train()

peft_model.save_pretrained(LORA_ADAPTER_PATH)
tokenizer.save_pretrained(LORA_ADAPTER_PATH)
print(f'\nLoRA adapter saved → {LORA_ADAPTER_PATH}')
print('Saved files:', os.listdir(LORA_ADAPTER_PATH))

## 3 · BERTScore Evaluation


In [ ]:
from bert_score import score as bert_score_fn
import re, torch, string
from tqdm import tqdm
from datasets import load_from_disk

peft_model.eval()

eos_ids = list(set(filter(None, [
    tokenizer.convert_tokens_to_ids("<end_of_turn>"),
    tokenizer.eos_token_id,
])))
print("EOS token ids:", eos_ids)


def normalize_text(text: str) -> str:
    """Lowercase and strip extra whitespace.
    BERTScore is context-aware so we intentionally keep punctuation —
    stripping it (as ROUGE eval did) removes signal the model can use.
    """
    return re.sub(r"\s+", " ", text.lower().strip())


def generate_answer(question: str) -> str:
    """Run beam-search inference on the fine-tuned LoRA model."""
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512,
    ).to(peft_model.device)

    with torch.no_grad():
        gen = peft_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            num_beams=4,              # beam search gives more coherent outputs
            early_stopping=True,
            no_repeat_ngram_size=3,
            repetition_penalty=1.15,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_ids = gen[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    raw = re.sub(r"<(end_of_turn|start_of_turn|bos|eos)>.*$", "", raw, flags=re.DOTALL).strip()
    return raw


def evaluate_dataset_bert(dataset_path: str, name: str):
    """Evaluate a TOFU split with BERTScore (Precision, Recall, F1).

    BERTScore uses contextual embeddings from a pre-trained transformer
    (roberta-large) to compare prediction and reference at the token level,
    capturing semantic similarity that ROUGE misses (synonyms, paraphrases,
    word-order variations).

    rescale_with_baseline=True maps raw cosine scores to a human-readable
    [0, 1] range using language-specific baselines.

    RTX 4090 note: roberta-large + the 4-bit LoRA model fit comfortably in
    24 GB VRAM at batch_size=32; lower to 16 if you see OOM errors.
    """
    print(f"\n--- Evaluating: {name} ---")
    dataset = load_from_disk(dataset_path)
    preds, refs = [], []

    for sample in tqdm(dataset, desc=name):
        pred = generate_answer(sample["question"])
        preds.append(normalize_text(pred))
        refs.append(normalize_text(sample["answer"]))

    # ── Spot-check a few samples ─────────────────────────────────────────
    print("\n[Sample Predictions]")
    for i in range(min(3, len(preds))):
        print(f"  Q   : {dataset[i]['question'][:80]}")
        print(f"  Pred: {preds[i][:120]}")
        print(f"  Ref : {refs[i][:120]}")
        print()

    # ── BERTScore ─────────────────────────────────────────────────────────
    P, R, F1 = bert_score_fn(
        preds, refs,
        lang="en",
        model_type="roberta-large",
        batch_size=32,
        rescale_with_baseline=True,
        device="cuda" if torch.cuda.is_available() else "cpu",
        verbose=True,
    )

    results = {
        "precision": P.mean().item(),
        "recall":    R.mean().item(),
        "f1":        F1.mean().item(),
    }
    print(f"\n\u2705 BERTScore [{name}]: {results}")
    return results, preds, refs


DATA_DIR = "/root/data_splits"
forget_bert, forget_preds, forget_refs = evaluate_dataset_bert(
    f"{DATA_DIR}/forget_dataset", "Forget-05"
)
retain_bert, retain_preds, retain_refs = evaluate_dataset_bert(
    f"{DATA_DIR}/retain_dataset", "Retain-95"
)

print("\n=== Summary ===")
print(f"Forget-05  BERTScore-F1 : {forget_bert['f1']:.4f}")
print(f"Retain-95  BERTScore-F1 : {retain_bert['f1']:.4f}")
print(f"Forget-05  Precision    : {forget_bert['precision']:.4f}  |  Recall : {forget_bert['recall']:.4f}")
print(f"Retain-95  Precision    : {retain_bert['precision']:.4f}  |  Recall : {retain_bert['recall']:.4f}")


## 4 · (Optional) Paraphrase & Perturbed-Answer Evaluation

TOFU provides `paraphrased_question` and `paraphrased_answer` columns which are used
in the official unlearning evaluation to measure **answer robustness** and
**truthfulness**. Run this cell after the main evaluation if needed.


In [ ]:
def evaluate_paraphrase_bert(dataset_path: str, name: str):
    """Evaluate paraphrased questions vs paraphrased reference answers using BERTScore.

    This mirrors the official TOFU robustness evaluation but replaces
    ROUGE with BERTScore for richer semantic comparison.
    """
    print(f"\n--- Paraphrase eval: {name} ---")
    dataset = load_from_disk(dataset_path)
    # Keep only rows that have paraphrase fields populated
    dataset = dataset.filter(
        lambda x: bool(x.get("paraphrased_question")) and bool(x.get("paraphrased_answer"))
    )
    if len(dataset) == 0:
        print("  No paraphrase data found — skipping.")
        return {}

    preds, refs = [], []
    for sample in tqdm(dataset, desc=f"{name} (paraphrase)"):
        pred = generate_answer(sample["paraphrased_question"])
        preds.append(normalize_text(pred))
        refs.append(normalize_text(sample["paraphrased_answer"]))

    P, R, F1 = bert_score_fn(
        preds, refs,
        lang="en",
        model_type="roberta-large",
        batch_size=32,
        rescale_with_baseline=True,
        device="cuda" if torch.cuda.is_available() else "cpu",
        verbose=True,
    )
    results = {
        "precision": P.mean().item(),
        "recall":    R.mean().item(),
        "f1":        F1.mean().item(),
    }
    print(f"\u2705 Paraphrase BERTScore [{name}]: {results}")
    return results


forget_para_bert = evaluate_paraphrase_bert(f"{DATA_DIR}/forget_dataset", "Forget-05")
retain_para_bert = evaluate_paraphrase_bert(f"{DATA_DIR}/retain_dataset", "Retain-95")


## 5 · Upload Adapter to HF Hub

In [ ]:
HF_UPLOAD_REPO    = 'Novaspree/tofu-Gemma3-adapter'   # ← change to your HF repo
LORA_ADAPTER_PATH = '/root/lora_adapter'

try:
    print(f'Uploading adapter to HF Hub: {HF_UPLOAD_REPO} ...')
    peft_model.push_to_hub(HF_UPLOAD_REPO, private=False)
    tokenizer.push_to_hub(HF_UPLOAD_REPO)
    print('✅ Upload complete!')
    print(f'Load with: PeftModel.from_pretrained(base_model, "{HF_UPLOAD_REPO}")')
except Exception as e:
    print(f'❌ Error: {e}')